# MLX Encoder-Decoder Transformer for Translation

This notebook is a rewrite and extension of the original `s2sTransformer.ipynb`. The original notebook used TensorFlow/Keras abstractions for text vectorization, model fitting, and saving. This version keeps the same sequence-to-sequence translation goal, but rebuilds the pipeline in MLX so the data preparation, Transformer internals, training loop, decoding strategy, and checkpoint format are explicit.

The model is not a chat-style LLM, but it uses the same core Transformer ideas used by many LLMs: token embeddings, positional information, self-attention, cross-attention, feed-forward layers, residual connections, normalization, cross-entropy training, and autoregressive decoding.

The high-level flow is:

1. Load and filter English/French sentence pairs from a public parallel corpus.
2. Train SentencePiece tokenizers so text becomes subword token IDs.
3. Build encoder inputs, shifted decoder inputs, and next-token targets.
4. Define the encoder-decoder Transformer directly in MLX.
5. Train with masked cross-entropy, token accuracy, validation monitoring, early stopping, and a Transformer-style warmup learning-rate schedule.
6. Translate new sentences with beam search.
7. Save and restore the model together with the tokenizer files and architecture configuration.


## Improvements Over the Original Notebook

The original notebook established the baseline idea: load sentence pairs, create shifted decoder targets, train an encoder-decoder model, and test translations. The rewrite extends that work in several ways that are useful for a technical reviewer to notice:

- **Framework migration:** replaces TensorFlow/Keras model and training abstractions with an MLX implementation aimed at Apple Silicon. The rewrite exposes the model call path, gradient step, optimizer update, validation loop, and checkpoint loading instead of delegating them to `model.fit()` and Keras serialization.

- **More reproducible data source:** moves from local absolute file paths to Hugging Face OPUS datasets, then builds a clean DataFrame from the nested translation records. This makes the notebook easier to rerun outside the original machine.

- **Subword tokenization:** replaces Keras `TextVectorization` with separately trained SentencePiece tokenizers for English and French. This improves handling of rare words and keeps source and target vocabularies independent.

- **Cleaner sequence construction:** separates encoder IDs, shifted decoder IDs, and target IDs. Padding, start, and end tokens are represented explicitly so masking and generation are easier to reason about.

- **Direct Transformer implementation:** defines positional encodings, encoder blocks, decoder blocks, causal masking, padding masking, residual connections, layer normalization, and vocabulary projection in code.

- **Training instrumentation:** adds masked loss, masked token accuracy, validation evaluation, early stopping, learning-rate warmup, and history plotting.

- **Better inference:** upgrades from greedy next-token decoding to beam search with length normalization and simple token suppression for cleaner outputs.

- **Portability:** saves model weights, tokenizers, and configuration together, avoiding the absolute-path Keras weight/model saves used in the original notebook.



In [30]:
#%pip install mlx pandas numpy pyarrow tqdm plotly

## Environment Check

Before defining the model, this cell checks that MLX is available and shows which device MLX will use. On Apple Silicon, MLX can run array operations efficiently on the GPU through Apple's unified memory system.


In [31]:
import mlx.core as mx

print(mx.__version__)
print(mx.default_device())

0.31.2
Device(gpu, 0)


## Imports and Reproducibility

These imports cover three parts of the notebook: data handling with pandas and NumPy, visualization with Plotly, and model building/training with MLX. The random seed helps make dataset splits and parameter initialization more repeatable.


In [32]:
import pandas as pd
import numpy as np

import plotly.express as px

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

## Dataset Dependency

The notebook uses Hugging Face `datasets` to download parallel translation corpora. A parallel corpus means each example contains a source sentence and its corresponding target-language translation.


In [33]:
%pip install datasets

/Users/chrisdreger/Desktop/LLM_Translation/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## Load an Example Parallel Corpus

This cell loads the OPUS Books English/French dataset. The next cell uses OPUS-100 instead, which gives a larger pool of sentence pairs for training.


In [34]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr")

## Build the Working DataFrame

The original notebook read from a machine-specific CSV path, which made the experiment difficult to reproduce elsewhere. This version pulls OPUS-100 from Hugging Face, flattens each nested translation record into simple English and French columns, and then filters noisy chapter/title-style rows and unusually long examples.

The filtering is intentionally lightweight: it removes obvious outliers while keeping the notebook focused on the modeling pipeline rather than corpus cleaning. The final shuffle makes the later train/validation split less dependent on the source dataset order.


In [35]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("Helsinki-NLP/opus-100", "en-fr")

rows = [
    {
        "English words/sentences": item["translation"]["en"],
        "French words/sentences": item["translation"]["fr"],
    }
    for item in dataset["train"].select(range(120_000))
]

easy_dataset = pd.DataFrame(rows)

print("Before filtering:", len(easy_dataset))

# Remove chapter/title style rows
bad_patterns = [
    "CHAPTER",
    "Chapter",
    "CHAPITRE",
    "Chapitre",
]

mask = ~easy_dataset["English words/sentences"].str.contains(
    "|".join(bad_patterns),
    na=False,
)

mask &= ~easy_dataset["French words/sentences"].str.contains(
    "|".join(bad_patterns),
    na=False,
)

easy_dataset = easy_dataset[mask]

# Remove unusually long examples
easy_dataset = easy_dataset[
    easy_dataset["English words/sentences"].str.len() < 250
]

easy_dataset = easy_dataset[
    easy_dataset["French words/sentences"].str.len() < 300
]

# Shuffle after filtering
easy_dataset = easy_dataset.sample(
    frac=1,
    random_state=42,
).reset_index(drop=True)

print("After filtering:", len(easy_dataset))

easy_dataset.head()

Before filtering: 120000
After filtering: 110898


,English words/sentences,French words/sentences
0,How?,Comment ?
1,It's your sister.,Ta sœur.
2,Why are we celebrating that?,Pourquoi en faire un fromage?
3,Arrangement of tyre markings,Schéma des inscriptions du pneumatique
4,"Baby brother, I got you a present!","Petit frère, j'ai un cadeau !"


## Inspect the Dataset Table

This quick check confirms the columns, row count, and missing-value status before the notebook starts deriving sentence lengths and train/validation splits.


In [36]:
easy_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 110898 entries, 0 to 110897
Data columns (total 2 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   English words/sentences  110898 non-null  str  
 1   French words/sentences   110898 non-null  str  
dtypes: str(2)
memory usage: 17.2 MB


## Sentence Length Exploration

Transformers process fixed-size batches, so very long examples can make training slow and memory-heavy. These length statistics help choose a practical maximum sequence length and identify sentences that should be filtered out.


In [37]:
easy_dataset["English Words in Sentence"] = (
    easy_dataset["English words/sentences"].str.split().apply(len)
)

easy_dataset["French Words in Sentence"] = (
    easy_dataset["French words/sentences"].str.split().apply(len)
)

fig = px.histogram(
    easy_dataset,
    x=["English Words in Sentence", "French Words in Sentence"],
    labels={"variable": "Language", "value": "Words in Sentence"},
    marginal="box",
    barmode="group",
    height=540,
    width=840,
    title="English-French Dataset - Words in Sentence",
)

fig.update_layout(
    title_font_size=18,
    bargap=0.2,
    bargroupgap=0.1,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        xanchor="right",
        y=1.02,
        x=1,
    ),
    yaxis_title="Count",
)

fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'alignmentgroup': 'True',
              'bingroup': 'x',
              'hovertemplate': ('Language=English Words in Sent' ... '}<br>count=%{y}<extra></extra>'),
              'legendgroup': 'English Words in Sentence',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'English Words in Sentence',
              'offsetgroup': 'English Words in Sentence',
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('AQMFBAcFAwojHhwLBR0SBAEHDAkBAw' ... 'IGKwESCgQBDggPCQ8HCgMKBxoIFAED'),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'},
             {'alignmentgroup': 'True',
              'hovertemplate': 'Language=English Words in Sentence<br>Words in Sentence=%{x}<extra></extra>',
              'legendgroup': 'English Words in Sentence',
              'marker': {'color': '#636efa'},
              'name': 'English Words in Sentence',
              'notched': True,
              'offsetgroup': 'English Words in Sentence',
              'showlegend': False,
              'type': 'box',
              'x': {'bdata': ('AQMFBAcFAwojHhwLBR0SBAEHDAkBAw' ... 'IGKwESCgQBDggPCQ8HCgMKBxoIFAED'),
                    'dtype': 'i1'},
              'xaxis': 'x2',
              'yaxis': 'y2'},
             {'alignmentgroup': 'True',
              'bingroup': 'x',
              'hovertemplate': ('Language=French Words in Sente' ... '}<br>count=%{y}<extra></extra>'),
              'legendgroup': 'French Words in Sentence',
              'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
              'name': 'French Words in Sentence',
              'offsetgroup': 'French Words in Sentence',
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('AgIFBQYDBQkoJiANBzAUBAEECgQCAg' ... 'IGKQESCwIBDgUNCA8KCgIICSQJFgED'),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'},
             {'alignmentgroup': 'True',
              'hovertemplate': 'Language=French Words in Sentence<br>Words in Sentence=%{x}<extra></extra>',
              'legendgroup': 'French Words in Sentence',
              'marker': {'color': '#EF553B'},
              'name': 'French Words in Sentence',
              'notched': True,
              'offsetgroup': 'French Words in Sentence',
              'showlegend': False,
              'type': 'box',
              'x': {'bdata': ('AgIFBQYDBQkoJiANBzAUBAEECgQCAg' ... 'IGKQESCwIBDgUNCA8KCgIICSQJFgED'),
                    'dtype': 'i1'},
              'xaxis': 'x2',
              'yaxis': 'y2'}],
    'layout': {'bargap': 0.2,
               'bargroupgap': 0.1,
               'barmode': 'group',
               'height': 540,
               'legend': {'orientation': 'h',
                          'title': {'text': 'Language'},
                          'tracegroupgap': 0,
                          'x': 1,
                          'xanchor': 'right',
                          'y': 1.02,
                          'yanchor': 'bottom'},
               'template': '...',
               'title': {'font': {'size': 18}, 'text': 'English-French Dataset - Words in Sentence'},
               'width': 840,
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Words in Sentence'}},
               'xaxis2': {'anchor': 'y2', 'domain': [0.0, 1.0], 'matches': 'x', 'showgrid': True, 'showticklabels': False},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 0.7326], 'title': {'text': 'Count'}},
               'yaxis2': {'anchor': 'x2',
                          'domain': [0.7426, 1.0],
                          'matches': 'y2',
                          'showgrid': False,
                          'showline': False,
                          'showticklabels': False,
                          'ticks': ''}}
})

## Train/Validation Split

Here the cleaned sentence pairs are converted into NumPy arrays and split into training and validation sets. Training data updates the model weights; validation data estimates how well the model generalizes to examples it did not train on.


In [38]:
sentences_en = easy_dataset["English words/sentences"].to_numpy()
sentences_fr = easy_dataset["French words/sentences"].to_numpy()

valid_fraction = 0.1
valid_len = int(valid_fraction * len(easy_dataset))

sentences_en_train = sentences_en[:-valid_len]
sentences_fr_train = sentences_fr[:-valid_len]

sentences_en_valid = sentences_en[-valid_len:]
sentences_fr_valid = sentences_fr[-valid_len:]


## Train SentencePiece Tokenizers

Neural translation models do not consume raw strings. SentencePiece learns a compact subword vocabulary, then converts each sentence into integer token IDs.

This is a meaningful upgrade from the original Keras `TextVectorization` approach. Word-level or whitespace-driven tokenization struggles with rare words, inflections, punctuation variants, and names. Subword tokenization can represent unfamiliar words as smaller learned pieces, which is a better match for translation.

Separate English and French tokenizers are trained because the source and target languages have different word shapes and frequency distributions. The notebook reserves padding and unknown IDs in SentencePiece, then adds target-only start/end IDs later so autoregressive decoding has explicit boundary markers.


In [39]:
import sentencepiece as spm
from pathlib import Path

sp_dir = Path("spm_models")
sp_dir.mkdir(exist_ok=True)

with open(sp_dir / "en_train.txt", "w", encoding="utf-8") as f:
    for s in sentences_en_train:
        f.write(str(s).strip() + "\n")

with open(sp_dir / "fr_train.txt", "w", encoding="utf-8") as f:
    for s in sentences_fr_train:
        f.write(str(s).strip() + "\n")

source_vocab_size = 8000
target_vocab_size = 8000

spm.SentencePieceTrainer.train(
    input=str(sp_dir / "en_train.txt"),
    model_prefix=str(sp_dir / "sp_en"),
    vocab_size=source_vocab_size,
    model_type="unigram",
    pad_id=0,
    unk_id=1,
    bos_id=-1,
    eos_id=-1,
)

spm.SentencePieceTrainer.train(
    input=str(sp_dir / "fr_train.txt"),
    model_prefix=str(sp_dir / "sp_fr"),
    vocab_size=target_vocab_size,
    model_type="unigram",
    pad_id=0,
    unk_id=1,
    bos_id=-1,
    eos_id=-1,
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: spm_models/en_train.txt
  input_format: 
  model_prefix: spm_models/sp_en
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: -1
  eos_id: -1
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differentia

## Prepare Encoder and Decoder Text

The Transformer is trained with teacher forcing:

- The encoder reads the complete English sentence.
- The decoder receives the French sentence shifted right.
- The target is the same French sentence shifted left.

At this raw-text preview stage the shift is shown with literal `startofseq` and `endofseq` markers. In the actual tokenized training batches later in the notebook, those boundaries become reserved numeric `START_ID` and `END_ID` values. The important idea is the offset: at decoder position `t`, the model sees the previous target tokens and learns to predict the next one.


In [40]:
def prepare_input_and_target(sentences_en, sentences_fr):
    """
    Return:
        encoder_input: English sentences
        decoder_input: French sentences with start token
        target:        French sentences with end token
    """
    decoder_input = ["startofseq " + s for s in sentences_fr]
    target = [s + " endofseq" for s in sentences_fr]

    return sentences_en, decoder_input, target

## Create Raw Text Mini-Batches

This helper yields small groups of sentence pairs. Batching lets the model process many examples at once, which is much faster than training one sentence pair at a time.


In [41]:
def batch_sentences(sentences_en, sentences_fr, batch_size=32, shuffle=False, seed=None):
    indices = np.arange(len(sentences_en))

    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        en_batch = sentences_en[batch_idx].tolist()
        fr_batch = sentences_fr[batch_idx].tolist()

        yield prepare_input_and_target(en_batch, fr_batch)

## Preview a Training Batch

Before tokenizing, this preview is a sanity check: each batch should contain aligned English inputs, French decoder inputs, and French prediction targets.


In [42]:
batch = next(
    batch_sentences(
        sentences_en_train,
        sentences_fr_train,
        batch_size=4,
        shuffle=True,
        seed=42,
    )
)

encoder_input, decoder_input, target = batch

print(encoder_input[0])
print(decoder_input[0])
print(target[0])

It has consolidated Serb opinion behind Milosevic and weakened the opposition.
startofseq Ils ont rassemblé l'opinion publique serbe derrière Milosevic et ont affaibli l'opposition.
Ils ont rassemblé l'opinion publique serbe derrière Milosevic et ont affaibli l'opposition. endofseq


## Check the Shifted Decoder Format

This cell shows the exact input/target shift used in sequence-to-sequence training. If the decoder input starts with `<s>` and the target ends with `</s>`, the model is being trained to generate the translation from left to right.


In [43]:
example_batch = next(
    batch_sentences(
        sentences_en_train,
        sentences_fr_train,
        batch_size=4,
    )
)

encoder_input, decoder_input, target = example_batch

print("Encoder input:")
print(encoder_input[0])

print("\nDecoder input:")
print(decoder_input[0])

print("\nTarget:")
print(target[0])

Encoder input:
How?

Decoder input:
startofseq Comment ?

Target:
Comment ? endofseq


## Estimate Training Work

The number of batches determines how many optimizer updates happen in one epoch. This is also useful when interpreting the learning-rate warmup schedule, which is defined in update steps rather than epochs.


In [44]:
import math

num_batches = math.ceil(len(sentences_en_train) / 32)

print(num_batches)

3120


## Migration Note: Removed Keras Baseline Cells

The original notebook included a Keras bidirectional LSTM with attention before moving to a Keras Transformer. In this rewrite, those intermediate LSTM cells are intentionally removed so the notebook stays centered on the MLX Transformer implementation.

That removal is part of the project scope rather than a missing section: the MLX version implements the pieces Keras previously handled automatically, including tokenization boundaries, batch creation, masked loss, gradient updates, validation, early stopping, and inference decoding.


## Positional Encoding

Self-attention can compare every token with every other token, but by itself it has no notion of word order. Positional encodings add a deterministic position signal to token embeddings so the model can distinguish sentences with the same words in different orders.

This implementation ports the sinusoidal encoding from the original Transformer paper into MLX arrays. The encoding has shape `[1, max_sentence_len, embedding_size]`, so it broadcasts across the batch and can be added directly to token embeddings. The even-indexed channels use sine waves and the odd-indexed channels use cosine waves at different frequencies, giving each position a distinctive representation.


In [50]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np


class PositionalEncoding(nn.Module):
    def __init__(
        self,
        max_sentence_len=50,
        embedding_size=256,
    ):
        super().__init__()

        if embedding_size % 2 != 0:
            raise ValueError(
                "embedding_size must be even"
            )

        p, i = np.meshgrid(
            np.arange(max_sentence_len),
            np.arange(embedding_size // 2),
        )

        pos_emb = np.empty(
            (1, max_sentence_len, embedding_size),
            dtype=np.float32,
        )

        pos_emb[:, :, 0::2] = (
            np.sin(
                p / (10000 ** (2 * i / embedding_size))
            ).T
        )

        pos_emb[:, :, 1::2] = (
            np.cos(
                p / (10000 ** (2 * i / embedding_size))
            ).T
        )

        self.positional_embedding = mx.array(pos_emb)

    def __call__(self, x):
        seq_len = x.shape[1]

        return (
            x
            + self.positional_embedding[:, :seq_len]
        )

## Positional Encoding Shape Check

This small test confirms that positional encoding preserves the batch, sequence, and embedding dimensions. The model expects tensors shaped like `[batch, sequence_length, embedding_size]` throughout most of the network.


In [51]:
pos_enc = PositionalEncoding(
    max_sentence_len=15,
    embedding_size=256,
)

x = mx.zeros((2, 15, 256))

y = pos_enc(x)

print(y.shape)

(2, 15, 256)


## Transformer Building Blocks

The original Keras Transformer relied on high-level layer classes and `model.fit()`. Here the encoder and decoder blocks are written directly so the architecture is visible.

Each block combines attention with a feed-forward network:

- Multi-head attention lets the model represent several token relationships in parallel.
- The feed-forward network transforms each token representation independently after attention has mixed sequence context.
- Residual connections preserve the previous representation and improve gradient flow.
- Layer normalization stabilizes activations after each residual addition.

The decoder has two attention stages: masked self-attention over the partial French output, followed by cross-attention over the encoded English sentence.


In [52]:
class FeedForward(nn.Module):
    def __init__(self, embedding_size=256, n_units_dense=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_size, n_units_dense),
            nn.ReLU(),
            nn.Linear(n_units_dense, embedding_size),
        )

    def __call__(self, x):
        return self.net(x)


class Encoder(nn.Module):
    def __init__(
        self,
        embedding_size=256,
        n_attention_heads=8,
        n_units_dense=256,
    ):
        super().__init__()

        self.self_attention = nn.MultiHeadAttention(
            dims=embedding_size,
            num_heads=n_attention_heads,
        )

        self.feed_forward = FeedForward(
            embedding_size=embedding_size,
            n_units_dense=n_units_dense,
        )

        self.norm1 = nn.LayerNorm(embedding_size)
        self.norm2 = nn.LayerNorm(embedding_size)

    def __call__(self, x, mask=None):
        skip = x

        x = self.self_attention(
            x,
            x,
            x,
            mask=mask,
        )

        x = self.norm1(x + skip)

        skip = x
        x = self.feed_forward(x)

        return self.norm2(x + skip)


class Decoder(nn.Module):
    def __init__(
        self,
        embedding_size=256,
        n_attention_heads=8,
        n_units_dense=256,
    ):
        super().__init__()

        self.self_attention = nn.MultiHeadAttention(
            dims=embedding_size,
            num_heads=n_attention_heads,
        )

        self.cross_attention = nn.MultiHeadAttention(
            dims=embedding_size,
            num_heads=n_attention_heads,
        )

        self.feed_forward = FeedForward(
            embedding_size=embedding_size,
            n_units_dense=n_units_dense,
        )

        self.norm1 = nn.LayerNorm(embedding_size)
        self.norm2 = nn.LayerNorm(embedding_size)
        self.norm3 = nn.LayerNorm(embedding_size)

    def __call__(
        self,
        x,
        encoder_output,
        decoder_mask=None,
        encoder_mask=None,
    ):
        skip = x

        x = self.self_attention(
            x,
            x,
            x,
            mask=decoder_mask,
        )

        x = self.norm1(x + skip)

        skip = x

        x = self.cross_attention(
            x,
            encoder_output,
            encoder_output,
            mask=encoder_mask,
        )

        x = self.norm2(x + skip)

        skip = x
        x = self.feed_forward(x)

        return self.norm3(x + skip)

## Attention Masks

Attention masks are one of the easiest places to introduce silent translation bugs, so the shapes are made explicit here.

The padding mask has shape `[batch, 1, 1, seq_len]`. It prevents the attention mechanism from treating padding tokens as meaningful words. The causal mask has shape `[1, 1, target_len, target_len]`. It is lower triangular, so decoder position `t` can attend only to positions `<= t`.

The decoder combines both masks: padding is ignored, and future target positions are hidden. This makes training match inference, where the model must generate the French sentence left to right without seeing future tokens.


In [53]:
def make_padding_mask(token_ids):
    """
    token_ids shape:
        [batch, seq_len]

    returns mask shape:
        [batch, 1, 1, seq_len]
    """
    return token_ids[:, None, None, :] != 0


def make_causal_mask(seq_len):
    """
    returns shape:
        [1, 1, seq_len, seq_len]
    """
    mask = mx.tril(
        mx.ones((seq_len, seq_len), dtype=mx.bool_)
    )

    return mask[None, None, :, :]

## Full Encoder-Decoder Transformer

The model has two cooperating halves:

- The encoder reads English token IDs, adds positional encodings, and produces contextual source representations.
- The decoder reads shifted French token IDs, applies masked self-attention, then uses cross-attention to condition each French position on the encoded English sentence.

The final linear layer returns logits over the French vocabulary for every decoder position. The model returns raw logits rather than probabilities because MLX cross-entropy functions handle the numerically stable softmax operation internally.


In [ ]:
class Transformer(nn.Module):
    def __init__(
        self,
        source_vocab_size,
        target_vocab_size,
        max_sentence_len=50,
        embedding_size=256,
        n_encoder_decoder_blocks=1,
        n_attention_heads=8,
        n_units_dense=256,
    ):
        super().__init__()

        self.max_sentence_len = max_sentence_len
        self.target_vocab_size = target_vocab_size

        self.encoder_embedding = nn.Embedding(source_vocab_size, embedding_size)
        self.decoder_embedding = nn.Embedding(target_vocab_size, embedding_size)

        self.positional_encoding = PositionalEncoding(max_sentence_len, embedding_size)

        self.encoder_blocks = [
            Encoder(embedding_size, n_attention_heads, n_units_dense)
            for _ in range(n_encoder_decoder_blocks)
        ]

        self.decoder_blocks = [
            Decoder(embedding_size, n_attention_heads, n_units_dense)
            for _ in range(n_encoder_decoder_blocks)
        ]

        self.output_layer = nn.Linear(embedding_size, target_vocab_size)

    def __call__(self, encoder_input_ids, decoder_input_ids):
        # 1) Embed tokens and inject position information.
        encoder_x = self.positional_encoding(
            self.encoder_embedding(encoder_input_ids)
        )

        decoder_x = self.positional_encoding(
            self.decoder_embedding(decoder_input_ids)
        )

        # 2) Build masks for padding and causal decoding.
        encoder_pad_mask = make_padding_mask(encoder_input_ids)
        decoder_pad_mask = make_padding_mask(decoder_input_ids)
        decoder_causal_mask = make_causal_mask(decoder_input_ids.shape[1])

        decoder_mask = decoder_causal_mask & decoder_pad_mask

        # 3) Pass through encoder stack.
        x = encoder_x
        for encoder_block in self.encoder_blocks:
            x = encoder_block(x, mask=encoder_pad_mask)

        # 4) Pass through decoder stack with cross-attention to encoder output.
        encoder_output = x
        x = decoder_x
        for decoder_block in self.decoder_blocks:
            x = decoder_block(
                x,
                encoder_output,
                decoder_mask=decoder_mask,
                encoder_mask=encoder_pad_mask,
            )

        # 5) Project hidden states to target-vocabulary logits.
        return self.output_layer(x)

## Token ID Encoding Utilities

The trained SentencePiece models are loaded here, along with special token IDs for padding, unknown tokens, sequence start, and sequence end.

The helper functions convert text into fixed-length arrays:

- encoder IDs: English sentence tokens
- decoder IDs: French tokens starting with `START_ID`
- target IDs: French tokens ending with `END_ID`

SentencePiece is trained with padding and unknown IDs, while start/end IDs are added manually to the target vocabulary. That is why `target_vocab_size` is the French SentencePiece vocabulary size plus two. Padding to `max_sentence_len` keeps batches rectangular, and the loss/masks make sure those padding positions do not count as translation content.


In [ ]:
import sentencepiece as spm

sp_en = spm.SentencePieceProcessor()
sp_fr = spm.SentencePieceProcessor()

sp_en.load("spm_models/sp_en.model")
sp_fr.load("spm_models/sp_fr.model")

PAD_ID = 0
UNK_ID = 1

START_TOKEN = "<s>"
END_TOKEN = "</s>"

# Reserve two extra IDs in the target vocabulary for start/end tokens.
source_vocab_size = sp_en.get_piece_size()
target_vocab_size = sp_fr.get_piece_size() + 2
START_ID = target_vocab_size - 2
END_ID = target_vocab_size - 1


def encode_source_batch(sentences, max_len):
    """Tokenize source (English) sentences and pad to fixed length."""
    encoded = []

    for sentence in sentences:
        ids = sp_en.encode(str(sentence), out_type=int)
        ids = ids[:max_len]
        ids = ids + [PAD_ID] * (max_len - len(ids))
        encoded.append(ids)

    return mx.array(encoded, dtype=mx.int32)


def encode_target_batch(sentences, max_len, add_start=False, add_end=False):
    """Tokenize target (French) sentences and optionally add BOS/EOS."""
    encoded = []

    for sentence in sentences:
        ids = sp_fr.encode(str(sentence), out_type=int)

        if add_start:
            ids = [START_ID] + ids

        if add_end:
            ids = ids + [END_ID]

        ids = ids[:max_len]
        ids = ids + [PAD_ID] * (max_len - len(ids))

        encoded.append(ids)

    return mx.array(encoded, dtype=mx.int32)

## Loss Function

Training uses token-level cross-entropy. For each decoder position, the model predicts a distribution over the French vocabulary, and the loss penalizes it when the correct next token receives low probability.

The logits and targets are flattened from `[batch, seq_len, vocab]` and `[batch, seq_len]` into token-level rows before loss calculation. Padding tokens are then excluded from the average. Without that mask, the model could appear to improve by learning to predict padding, especially when many examples are shorter than `max_sentence_len`.


In [56]:
def loss_fn(model, encoder_ids, decoder_ids, target_ids):
    logits = model(encoder_ids, decoder_ids)

    loss = nn.losses.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        target_ids.reshape(-1),
        reduction="none",
    )

    mask = target_ids.reshape(-1) != PAD_ID

    loss = loss * mask

    return loss.sum() / mask.sum()

## Learning-Rate Warmup Schedule

Transformers are sensitive to large updates at the beginning of training. The warmup schedule starts with a small learning rate, increases it for the first several thousand steps, then gradually decays it.

This mirrors the original Transformer recipe and usually makes training more stable.


In [57]:
def get_lr(step, d_model=256, warmup_steps=4000):
    step = max(step, 1)

    return (d_model ** -0.5) * min(
        step ** -0.5,
        step * (warmup_steps ** -1.5),
    )

## One Training Step

A training step performs the full learning update that Keras previously hid inside `model.fit()`:

1. Set the current warmup-scheduled learning rate.
2. Run the model and compute masked cross-entropy.
3. Use `nn.value_and_grad` to backpropagate through the Transformer.
4. Ask Adam to update the model parameters.
5. Materialize MLX's lazy computations with `mx.eval` before reading metrics.
6. Report loss and masked token-level accuracy.

Making this loop explicit is useful for debugging and for demonstrating control over the optimization process.


In [ ]:
def train_step(model, optimizer, encoder_ids, decoder_ids, target_ids, lr):
    """Run one optimization step and return loss + token accuracy."""
    # Apply the warmup-scheduled learning rate for this update.
    optimizer.learning_rate = lr

    # Forward + backward pass through the current batch.
    loss, grads = loss_and_grad_fn(
        model,
        encoder_ids,
        decoder_ids,
        target_ids,
    )

    # Update model parameters with Adam.
    optimizer.update(model, grads)

    # Ensure parameters/state are materialized before metric computation.
    mx.eval(model.parameters(), optimizer.state)

    acc = accuracy_fn(
        model,
        encoder_ids,
        decoder_ids,
        target_ids,
    )

    return loss, acc

## Token-Level Accuracy

Accuracy counts how often the highest-scoring predicted token equals the target token. Padding positions are ignored for the same reason they are ignored in the loss: they are batch filler, not translation content.

This metric is useful for quick feedback, but translation quality should still be judged with generated examples or translation-specific metrics.


In [59]:
def accuracy_fn(model, encoder_ids, decoder_ids, target_ids):
    logits = model(
        encoder_ids,
        decoder_ids,
    )

    predictions = mx.argmax(
        logits,
        axis=-1,
    )

    mask = target_ids != PAD_ID

    correct = (predictions == target_ids) & mask

    return correct.sum() / mask.sum()

## Tokenized Batch Builder

This is the training batch function used by the main loop. It takes raw sentence strings, encodes them with SentencePiece, pads them to `max_sentence_len`, and returns MLX arrays ready for the Transformer.


In [ ]:
def make_batch(sentences_en, sentences_fr, batch_size=32, shuffle=True, seed=None):
    """Yield tokenized mini-batches for encoder input, decoder input, and targets."""
    indices = np.arange(len(sentences_en))

    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        en_batch = sentences_en[batch_idx]
        fr_batch = sentences_fr[batch_idx]

        # Encoder sees source sentence tokens.
        encoder_ids = encode_source_batch(
            en_batch,
            max_sentence_len,
        )

        # Decoder input is shifted right: starts with <s>.
        decoder_ids = encode_target_batch(
            fr_batch,
            max_sentence_len,
            add_start=True,
            add_end=False,
        )

        # Target is shifted left: ends with </s>.
        target_ids = encode_target_batch(
            fr_batch,
            max_sentence_len,
            add_start=False,
            add_end=True,
        )

        yield encoder_ids, decoder_ids, target_ids

## Validation Evaluation

Evaluation runs the model without applying gradient updates. It computes average validation loss and accuracy so the notebook can track whether training is improving generalization rather than only memorizing the training set.


In [61]:
def evaluate(model, sentences_en, sentences_fr, batch_size=32):
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for encoder_ids, decoder_ids, target_ids in make_batch(
        sentences_en,
        sentences_fr,
        batch_size=batch_size,
        shuffle=False,
    ):
        loss = loss_fn(model, encoder_ids, decoder_ids, target_ids)
        acc = accuracy_fn(model, encoder_ids, decoder_ids, target_ids)

        total_loss += float(loss)
        total_acc += float(acc)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

## Training Loop with Early Stopping

The training loop repeatedly creates shuffled batches, updates the model, and evaluates on validation data. It records loss, accuracy, validation loss, validation accuracy, and learning rate after each epoch.

The notebook keeps a deep copy of the best parameter tree seen so far and restores it after training. Early stopping is based on validation loss, so training stops when generalization stops improving rather than when the model merely continues fitting the training data.


In [ ]:
import copy

def train(
    model,
    optimizer,
    n_epochs=5,
    batch_size=32,
    patience=5,
    d_model=256,
    warmup_steps=4000,
):
    """Train model with LR warmup, validation monitoring, and early stopping."""
    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "lr": [],
    }

    best_val_loss = float("inf")
    best_params = None
    epochs_without_improvement = 0
    global_step = 0

    for epoch in range(n_epochs):
        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0

        # Training phase: shuffle each epoch for better mixing.
        for encoder_ids, decoder_ids, target_ids in make_batch(
            sentences_en_train,
            sentences_fr_train,
            batch_size=batch_size,
            shuffle=True,
            seed=42 + epoch,
        ):
            global_step += 1

            lr = get_lr(
                global_step,
                d_model=d_model,
                warmup_steps=warmup_steps,
            )

            loss, acc = train_step(
                model,
                optimizer,
                encoder_ids,
                decoder_ids,
                target_ids,
                lr,
            )

            total_loss += float(loss)
            total_acc += float(acc)
            n_batches += 1

        train_loss = total_loss / n_batches
        train_acc = total_acc / n_batches

        # Validation phase: no parameter updates.
        val_loss, val_acc = evaluate(
            model,
            sentences_en_valid,
            sentences_fr_valid,
            batch_size=batch_size,
        )

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)
        history["lr"].append(lr)

        print(
            f"Epoch {epoch + 1:02d} - "
            f"loss: {train_loss:.4f} - "
            f"accuracy: {train_acc:.4f} - "
            f"val_loss: {val_loss:.4f} - "
            f"val_accuracy: {val_acc:.4f} - "
            f"lr: {lr:.6f}"
        )

        # Track the best validation checkpoint.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_params = copy.deepcopy(model.parameters())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    if best_params is not None:
        model.update(best_params)
        print(f"Restored best model with val_loss: {best_val_loss:.4f}")

    return history

## Confirm Dataset Size

This prints the number of training examples available from the loaded corpus. It helps interpret epoch time and the number of optimizer steps per epoch.


In [63]:
print(len(dataset["train"]))

1000000


## Instantiate, Optimize, and Train the Transformer

This cell sets the key hyperparameters, creates the Transformer and Adam optimizer, then starts training.

Important knobs include:

- `max_sentence_len`: maximum encoded length retained for each sentence
- `embedding_size`: width of each token representation
- `n_encoder_decoder_blocks`: depth of the encoder and decoder stacks
- `n_attention_heads`: number of attention heads per attention layer
- `n_units_dense`: width of the feed-forward subnetwork inside each block
- `batch_size`: number of sentence pairs per optimizer step
- `warmup_steps`: number of optimizer steps used to ramp up the learning rate

The rewrite increases the practical sequence length relative to the original short Keras run and uses a two-block Transformer, which gives the model more room to learn sentence-level structure at the cost of more compute.


In [34]:
mx.random.seed(42)

max_sentence_len = 96

transformer = Transformer(
    source_vocab_size=source_vocab_size,
    target_vocab_size=target_vocab_size,
    max_sentence_len=max_sentence_len,
    embedding_size=256,
    n_encoder_decoder_blocks=2,
    n_attention_heads=8,
    n_units_dense=512,
)

optimizer = optim.Adam(learning_rate=1e-4)

loss_and_grad_fn = nn.value_and_grad(transformer, loss_fn)

history = train(
    transformer,
    optimizer,
    n_epochs=30,
    batch_size=64,
    patience=4,
    d_model=256,
    warmup_steps=4000,
)

Epoch 01 - loss: 5.8078 - accuracy: 0.1636 - val_loss: 4.7967 - val_accuracy: 0.2294 - lr: 0.000385
Epoch 02 - loss: 4.3390 - accuracy: 0.2813 - val_loss: 3.9373 - val_accuracy: 0.3142 - lr: 0.000771
Epoch 03 - loss: 3.5011 - accuracy: 0.3921 - val_loss: 3.2910 - val_accuracy: 0.4034 - lr: 0.000914
Epoch 04 - loss: 2.8693 - accuracy: 0.4761 - val_loss: 2.9868 - val_accuracy: 0.4482 - lr: 0.000791
Epoch 05 - loss: 2.5085 - accuracy: 0.5241 - val_loss: 2.8693 - val_accuracy: 0.4665 - lr: 0.000708
Epoch 06 - loss: 2.2590 - accuracy: 0.5595 - val_loss: 2.8027 - val_accuracy: 0.4797 - lr: 0.000646
Epoch 07 - loss: 2.0572 - accuracy: 0.5897 - val_loss: 2.8122 - val_accuracy: 0.4822 - lr: 0.000598
Epoch 08 - loss: 1.8846 - accuracy: 0.6173 - val_loss: 2.8526 - val_accuracy: 0.4851 - lr: 0.000559
Epoch 09 - loss: 1.7307 - accuracy: 0.6429 - val_loss: 2.9049 - val_accuracy: 0.4822 - lr: 0.000527
Epoch 10 - loss: 1.5899 - accuracy: 0.6680 - val_loss: 2.9905 - val_accuracy: 0.4787 - lr: 0.000500


## Display Training Curves

The training history plot makes it easier to see whether loss is decreasing, validation loss is flattening, or early stopping is likely to trigger.


In [35]:
import plotly.io as pio

pio.renderers.default = "browser"
fig.show()

## Plot Loss and Accuracy History

This converts the collected history into a DataFrame and plots the training/validation metrics across epochs. Smooth improvement usually means the model and data pipeline are wired correctly.


In [36]:
import pandas as pd
import plotly.express as px

history_df = pd.DataFrame(history)

fig = px.line(
    history_df,
    markers=True,
    title="MLX Transformer Training"
)

fig.show()

## Inspect the Trained Model

Printing the model is a quick architecture check: it confirms the configured number of encoder/decoder blocks and shows the modules that were trained.


In [37]:
print(transformer)
print(len(transformer.encoder_blocks))

Transformer(
  (encoder_embedding): Embedding(8000, 256)
  (decoder_embedding): Embedding(8002, 256)
  (positional_encoding): PositionalEncoding()
  (encoder_blocks.0): Encoder(
    (self_attention): MultiHeadAttention(
      (query_proj): Linear(input_dims=256, output_dims=256, bias=False)
      (key_proj): Linear(input_dims=256, output_dims=256, bias=False)
      (value_proj): Linear(input_dims=256, output_dims=256, bias=False)
      (out_proj): Linear(input_dims=256, output_dims=256, bias=False)
    )
    (feed_forward): FeedForward(
      (net): Sequential(
        (layers.0): Linear(input_dims=256, output_dims=512, bias=True)
        (layers.1): ReLU()
        (layers.2): Linear(input_dims=512, output_dims=256, bias=True)
      )
    )
    (norm1): LayerNorm(256, eps=1e-05, affine=True)
    (norm2): LayerNorm(256, eps=1e-05, affine=True)
  )
  (encoder_blocks.1): Encoder(
    (self_attention): MultiHeadAttention(
      (query_proj): Linear(input_dims=256, output_dims=256, bias=Fal

## Decoding Helpers

After training, the model no longer receives the full French sentence. Instead, it starts with `<s>` and repeatedly predicts the next token.

These helpers convert generated token IDs back into readable French text and remove special tokens such as padding, start, and end markers.


In [64]:
def decode_tokens(token_ids):
    clean_ids = []

    for token_id in token_ids:
        token_id = int(token_id)

        if token_id == PAD_ID:
            continue

        if token_id == START_ID:
            continue

        if token_id == END_ID:
            break

        clean_ids.append(token_id)

    return sp_fr.decode(clean_ids)

## Beam Search Translation

Greedy decoding keeps only the single best next token at each step. Beam search keeps several partial translations alive, which helps recover from locally tempting choices that lead to worse full sentences.

At every decoding step, the model scores possible next tokens, expands the best beams, and stops when beams produce the end token or reach the maximum length. The ranking uses length-normalized log probability so very short translations do not dominate simply because they accumulate fewer negative log-probability terms.

The decoder also suppresses padding, unknown, repeated, and duplicate start tokens during generation. Those rules are simple heuristics, but they make interactive examples cleaner and expose where production systems often add decoding constraints.


In [ ]:
def suppress_token(logits, token_id):
    """Force a token to be effectively unselectable during decoding."""
    vocab_positions = mx.arange(logits.shape[0])

    return mx.where(
        vocab_positions == token_id,
        mx.array(-1e9, dtype=logits.dtype),
        logits,
    )


def translate_beam(model, sentence_en, beam_width=4):
    """Translate one English sentence to French with beam search."""
    encoder_ids = encode_source_batch(
        [sentence_en],
        max_sentence_len,
    )

    # Each beam item: (token_sequence, cumulative_log_probability)
    beams = [
        ([START_ID], 0.0)
    ]

    # Autoregressive decoding up to max length.
    for _ in range(max_sentence_len):
        candidates = []

        for tokens, score in beams:
            # Keep completed beams as-is.
            if tokens[-1] == END_ID:
                candidates.append((tokens, score))
                continue

            decoder_ids = mx.array(
                [
                    tokens + [PAD_ID] * (max_sentence_len - len(tokens))
                ],
                dtype=mx.int32,
            )

            logits = model(encoder_ids, decoder_ids)
            next_logits = logits[0, len(tokens) - 1]

            # Block undesirable tokens for cleaner outputs.
            next_logits = suppress_token(next_logits, PAD_ID)
            next_logits = suppress_token(next_logits, UNK_ID)
            next_logits = suppress_token(next_logits, START_ID)

            # Light repetition penalty for tokens already used multiple times.
            for seen_token in set(tokens):
                if tokens.count(seen_token) >= 2:
                    next_logits = suppress_token(next_logits, seen_token)

            log_probs = nn.log_softmax(next_logits, axis=-1)
            top_ids = mx.argsort(-log_probs)[:beam_width]

            for token_id in top_ids:
                token_id = int(token_id)
                token_score = float(log_probs[token_id])

                candidates.append(
                    (
                        tokens + [token_id],
                        score + token_score,
                    )
                )

        # Length-normalized ranking prevents very short beams from dominating.
        candidates = sorted(
            candidates,
            key=lambda x: x[1] / (len(x[0]) ** 0.7),
            reverse=True,
        )

        # Keep only top-k candidates.
        beams = candidates[:beam_width]

        # Stop if every beam has already finished.
        if all(tokens[-1] == END_ID for tokens, _ in beams):
            break

    best_tokens, best_score = beams[0]

    return decode_tokens(best_tokens)

## Try Example Translations

These examples run the trained model in inference mode. The quality depends heavily on dataset size, tokenizer settings, model capacity, and training time.


In [66]:
print(translate_beam(transformer, "Hi, what's your name?", beam_width=4))
print(translate_beam(transformer, "I wish Tom was here", beam_width=4))
print(translate_beam(transformer, "She ordered him to do it", beam_width=4))
print(translate_beam(transformer, "Good morning", beam_width=4))
print(translate_beam(transformer, "Thank you", beam_width=4))

NameError: name 'transformer' is not defined

## Save the Trained Model

MLX model parameters are nested trees, so this cell flattens them before saving to `transformer_weights.npz`. The directory also stores the SentencePiece tokenizer models and a JSON configuration with architecture and special-token values.

This is a deliberate improvement over the original notebook's absolute-path Keras saves. A future reader can reconstruct the model from the saved folder without knowing the training machine's directory layout, and the tokenizer files travel with the weights that depend on them.


In [ ]:
from pathlib import Path
from mlx.utils import tree_flatten
import mlx.core as mx
import json
import shutil

save_dir = Path("saved_translation_model")
save_dir.mkdir(exist_ok=True)

# 1. Save Transformer weights
weights = dict(tree_flatten(transformer.parameters()))

mx.savez(
    str(save_dir / "transformer_weights.npz"),
    **weights,
)

# 2. Save SentencePiece tokenizers
shutil.copy("spm_models/sp_en.model", save_dir / "sp_en.model")
shutil.copy("spm_models/sp_fr.model", save_dir / "sp_fr.model")

# 3. Save model config
config = {
    "source_vocab_size": source_vocab_size,
    "target_vocab_size": target_vocab_size,
    "max_sentence_len": max_sentence_len,
    "embedding_size": 256,
    "n_encoder_decoder_blocks": 2,
    "n_attention_heads": 8,
    "n_units_dense": 512,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "start_id": START_ID,
    "end_id": END_ID,
}

with open(save_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved model to: {save_dir.resolve()}")

Saved model to: /Users/chrisdreger/Desktop/LLM_Translation/saved_translation_model


## Load a Saved Checkpoint

This helper reverses the save process: it reads the config, rebuilds the Transformer architecture, loads the saved weights, and restores the SentencePiece tokenizers.

The architecture must be reconstructed with the same vocabulary sizes, sequence length, embedding width, block count, attention-head count, and feed-forward width used during training. If any of those values drift, the loaded parameter shapes will no longer match the model. Keeping them in `config.json` makes inference reproducible and avoids hidden notebook state.


In [67]:
def load_translation_model(model_dir="saved_translation_model"):
    import json
    from pathlib import Path
    import sentencepiece as spm
    import mlx.core as mx
    from mlx.utils import tree_unflatten

    model_dir = Path(model_dir)

    with open(model_dir / "config.json") as f:
        config = json.load(f)

    sp_en = spm.SentencePieceProcessor(
        model_file=str(model_dir / "sp_en.model")
    )

    sp_fr = spm.SentencePieceProcessor(
        model_file=str(model_dir / "sp_fr.model")
    )

    transformer = Transformer(
        source_vocab_size=config["source_vocab_size"],
        target_vocab_size=config["target_vocab_size"],
        max_sentence_len=config["max_sentence_len"],
        embedding_size=config["embedding_size"],
        n_encoder_decoder_blocks=config["n_encoder_decoder_blocks"],
        n_attention_heads=config["n_attention_heads"],
        n_units_dense=config["n_units_dense"],
    )

    weights = mx.load(str(model_dir / "transformer_weights.npz"))

    transformer.update(
        tree_unflatten(
            list(weights.items())
        )
    )

    mx.eval(transformer.parameters())

    return transformer, sp_en, sp_fr, config

## Restore Model for Inference

This cell loads the saved checkpoint and resets the special token IDs from the saved config. Once this runs, the same `translate_beam` function can translate new English sentences using the restored weights.


In [68]:
transformer, sp_en, sp_fr, config = load_translation_model(
    "saved_translation_model"
)

PAD_ID = config["pad_id"]
UNK_ID = config["unk_id"]
START_ID = config["start_id"]
END_ID = config["end_id"]
max_sentence_len = config["max_sentence_len"]

## Example Translations

These examples run the trained model in inference mode. The quality depends heavily on dataset size, tokenizer settings, model capacity, and training time.


In [71]:
print(translate_beam(transformer, "Harrison is my boyfriend", beam_width=4))
print(translate_beam(transformer, "He lives in London and works in theatre", beam_width=4))

Harrison est mon petit copain.
Il vit dans la famille et travaille dans le pays.
